# TP tracking, human review, and POD classification

Run these cells in order in Anaconda/Jupyter. The first **20 TP positions are manually clicked**; these provide fixed appearance templates for tracking every selected frame. Correct or confirm flagged positions, save 16-frame reference sheets, and define three vertical-position categories before training the two classifiers.

**Review controls:** click the correct TP then **Enter** to correct; **A** to confirm the displayed tracked point is already correct; **S** to leave it unresolved and move on. Buttons provide the same actions. A confirmation is saved separately from a manual correction.

Only training TP positions define the mean ± one population SD band. POD uses images, not TP coordinates, as classifier inputs. Model selection uses validation; final predictions use a separate chronological test block. No experimental accuracy is claimed by this unexecuted source notebook.

## 1. Set parameters

Keep this notebook beside `schlieren_qml` and `configs`. Install missing dependencies with `%pip install -r requirements.txt`, then restart the kernel. After replacing an older project, restart the kernel so the new review functions load.

Your POD crop is 350×350 original pixels, becoming 175×175 after binning by two. TP annotation and tracking always use original full-resolution coordinates. Enable only the stage you intend to run, then rerun this parameter cell and that stage. A stage with missing prerequisites prints what to do next.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, shutil, csv, hashlib
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "schlieren_qml" / "triple_point.py").is_file():
    raise FileNotFoundError("Open this notebook inside the updated Hemisphere-QML project folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from schlieren_qml.triple_point import (
    launch_tp_annotation, track_tp, apply_tp_review,
    tp_categories, tp_gallery, _sequence,
)
from schlieren_qml.tp_reference import export_tp_montages
from schlieren_qml.tp_evaluation import export_tp_test_predictions
from schlieren_qml.pod import load_pod
from schlieren_qml.figures import plot_spatial_modes
from schlieren_qml.notebook import (
    resolve_notebook_config, run_from_notebook,
    display_saved_figures, preview_notebook_roi,
)

IMAGE_DIRECTORY = Path(r"H:\FSU\Hemisphere\POD_QuantCom\20000Images")
START_FRAME = 0
FRAME_LIMIT = 20000
FPS = 51032.0
POD_ROI = [400, 160, 750, 510]
POD_BIN_FACTOR = 2
MODE_COLOR_VMAX = 0.009

TP_WORK_DIRECTORY = PROJECT_DIR / "results-TP" / "tp_measurement"
ANNOTATION_CSV = TP_WORK_DIRECTORY / "tp_manual.csv"
REVIEW_CSV = TP_WORK_DIRECTORY / "tp_review.csv"
TRACK_DIRECTORY = TP_WORK_DIRECTORY / "tracking"
TRACKS_CSV = TRACK_DIRECTORY / "tp_tracks.csv"
CATEGORY_DIRECTORY = TP_WORK_DIRECTORY / "categories"
REFERENCE_DIRECTORY = TP_WORK_DIRECTORY / "reference_sheets"
MANUAL_COUNT = 20
DESKTOP_BACKEND = "TkAgg"

PATCH_SIZE = 25
SEARCH_RADIUS = 20
TP_SEARCH_ROI = None
MIN_TRACK_SCORE = 0.75
MIN_PEAK_MARGIN = 0.05
MAX_TEMPLATES = 5
OVERWRITE_TRACKS = True
OVERWRITE_CATEGORIES = True
OVERWRITE_REFERENCE_SHEETS = True

# Classifier cost controls. All selected images are tracked and projected.
# These caps apply only to matched classifier training/validation/test samples.
CLASSIFIER_TRAIN_FRAMES = 512
CLASSIFIER_VALIDATION_FRAMES = 512
CLASSIFIER_TEST_FRAMES = 256
POD_RANK = 24
POD_FEATURES = 4
TRAIN_FRACTION = 0.6
VALIDATION_FRACTION = 0.2
GAP_FRAMES = 128

# 4x4 = 16 frames per page; 20,000 frames produce 1,250 pages.
# Larger tiles preserve more visible detail but use more disk space.
REFERENCE_TILE_WIDTH = 384
OVERLAY_SEED = 42
FLAGGED_ONLY = False
REVIEW_ALL_TRACKS = False  # True: inspect/correct even automatically accepted points.

# Run one stage at a time. Missing prerequisites prevent later stages starting.
OPEN_ANNOTATION_WINDOW = False
RUN_TRACKING = False
OPEN_CORRECTION_WINDOW = False
EXPORT_REFERENCE_SHEETS = False
EXPORT_TP_CATEGORIES = False
RUN_POD_AND_CLASSIFIERS = True
OVERWRITE_POD_RESULTS = False

config = json.loads((PROJECT_DIR / "configs" / "full.json").read_text())
config.update(
    image_directory=str(IMAGE_DIRECTORY), start=START_FRAME,
    limit=FRAME_LIMIT, stride=1, fps=FPS, roi=POD_ROI,
    bin_factor=POD_BIN_FACTOR, labels_csv=None, rank=POD_RANK,
    train_fraction=TRAIN_FRACTION, validation_fraction=VALIDATION_FRACTION,
    gap_frames=GAP_FRAMES,
)
config["classifier"].update(
    n_features=POD_FEATURES, max_train=CLASSIFIER_TRAIN_FRAMES,
    max_validation=CLASSIFIER_VALIDATION_FRAMES,
    max_test=CLASSIFIER_TEST_FRAMES,
)
config["figures"].update(
    mode_color_vmax=MODE_COLOR_VMAX, mode_color_percentile=None,
)
print("Annotations:", ANNOTATION_CSV)
print("Review decisions:", REVIEW_CSV)
print("Tracking:", TRACK_DIRECTORY)
print("Reference sheets:", REFERENCE_DIRECTORY)
print("Categories:", CATEGORY_DIRECTORY)
print("POD mode figure limits:", -MODE_COLOR_VMAX, MODE_COLOR_VMAX)

## 2. Check inputs and progress

Run after setting parameters. Missing seeds are reported explicitly. Keep the same annotation path for initial marking and later corrections. Do not mix painted/annotated copies into the TIFF source folder.

In [ ]:
def read_rows(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))

def missing_seed_names():
    paths, _ = _sequence(config)
    saved = {r["filename"] for r in read_rows(ANNOTATION_CSV)} if ANNOTATION_CSV.is_file() else set()
    return [p.name for p in paths[:MANUAL_COUNT] if p.name not in saved]

def current_tracking_ready(require_resolved=False):
    manifest_path = TRACK_DIRECTORY / "tp_tracking_manifest.json"
    if not TRACKS_CSV.is_file() or not manifest_path.is_file():
        print("No tracking saved. Complete initial annotations, then run tracking.")
        return False
    tracking_metadata = json.loads(manifest_path.read_text(encoding="utf-8"))
    saved = tracking_metadata["config"]
    keys = ("image_directory", "start", "stride", "limit", "fps", "roi", "bin_factor",
            "train_fraction", "validation_fraction", "gap_frames", "rank", "classifier")
    changed = [key for key in keys if saved.get(key) != config.get(key)]
    h, w = tracking_metadata["original_shape"]
    expected_tracking = dict(
        manual_count=MANUAL_COUNT, patch_size=PATCH_SIZE,
        search_radius=SEARCH_RADIUS,
        search_roi=TP_SEARCH_ROI if TP_SEARCH_ROI is not None else [0, 0, w, h],
        min_score=MIN_TRACK_SCORE, min_margin=MIN_PEAK_MARGIN,
    )
    changed += [key for key, value in expected_tracking.items()
                if tracking_metadata.get(key) != value]
    if len(tracking_metadata["template_filenames"]) != min(MAX_TEMPLATES, MANUAL_COUNT):
        changed.append("MAX_TEMPLATES")
    if changed:
        print("Parameters differ from saved tracking:", ", ".join(changed))
        print("Restore those parameters or rerun tracking before continuing.")
        return False
    rows = read_rows(TRACKS_CSV)
    pending = sum(r["accepted"] != "True" for r in rows)
    print(f"Tracked: {len(rows):,}; unresolved: {pending:,}")
    if require_resolved and pending:
        print("Review flagged points: A confirms correct; click+Enter corrects; S leaves unresolved.")
        return False
    return True

def refresh_original_modes(result_directory):
    result_directory = Path(result_directory)
    figures = result_directory / "figures"
    options = dict(config["figures"])
    frame_info = json.loads((result_directory / "frames.json").read_text())
    options.update(mode_color_vmax=MODE_COLOR_VMAX, mode_color_percentile=None,
                   bin_factor=frame_info.get("bin_factor", 1))
    scale_path = figures / "04_spatial_modes_color_scale.json"
    scale = json.loads(scale_path.read_text()) if scale_path.is_file() else {}
    if scale.get("vmax") != MODE_COLOR_VMAX or scale.get("vmin") != -MODE_COLOR_VMAX:
        plot_spatial_modes(load_pod(result_directory / "pod_model.npz"),
                           tuple(frame_info["image_shape"]), figures, config=options)
        notes_path = figures / "FIGURE_NOTES.md"
        if notes_path.is_file():
            note = (figures / "04_spatial_modes_notes.md").read_text().strip()
            lines = [note if line.startswith("04:") else line
                     for line in notes_path.read_text().splitlines()]
            if not any(line.startswith("04:") for line in lines):
                lines.append(note)
            notes_path.write_text("\n".join(lines) + "\n")
    print("Original spatial-mode figure limits:", -MODE_COLOR_VMAX, MODE_COLOR_VMAX)

def matching_model_results(result_directory, expected_config):
    result_directory = Path(result_directory)
    saved_config_path = result_directory / "config_used.json"
    saved_definition_path = result_directory / "tp_category_definition.json"
    if not saved_config_path.is_file() or not saved_definition_path.is_file():
        print("Saved model provenance is missing. Rerun the model stage.")
        return False
    used = json.loads(saved_config_path.read_text(encoding="utf-8"))
    saved_definition = json.loads(saved_definition_path.read_text(encoding="utf-8"))
    current_definition = json.loads(
        (CATEGORY_DIRECTORY / "tp_category_definition.json").read_text(encoding="utf-8")
    )
    digest = hashlib.sha256(TRACKS_CSV.read_bytes()).hexdigest()
    if any(d.get("tracks_sha256") != digest for d in (saved_definition, current_definition)):
        print("Saved models use different TP measurements. Re-export categories and rerun models.")
        return False
    changed = [key for key in set(used) | set(expected_config)
               if key != "figures" and used.get(key) != expected_config.get(key)]
    if changed:
        print("Saved models use different parameters:", ", ".join(sorted(changed)))
        print("Rerun the model stage before reporting this experiment.")
        return False
    return True

if IMAGE_DIRECTORY.is_dir():
    preview, summary = preview_notebook_roi(config)
    display(summary)
    display(preview)
    plt.close(preview)
    missing = missing_seed_names()
    print(f"Initial TP annotations missing: {len(missing)}")
    for name in missing[:20]:
        print("  ", name)
    current_tracking_ready()
else:
    print("Image folder not found. Check IMAGE_DIRECTORY before running stages.")

## 3. Mark the initial 20 TP positions

Set `OPEN_ANNOTATION_WINDOW=True`, rerun parameters, then this cell. Click TP and press Enter on each image. Backspace revisits the previous image. Closing preserves saved points; reopening resumes missing seeds. This stage deliberately has no flagged-review filter, so it can repair missing initial annotations.

In [ ]:
if OPEN_ANNOTATION_WINDOW:
    if IMAGE_DIRECTORY.is_dir():
        launch_tp_annotation(config, ANNOTATION_CSV,
                             count=MANUAL_COUNT, backend=DESKTOP_BACKEND)
        print("Missing initial annotations:", len(missing_seed_names()))
    else:
        print("Set a valid IMAGE_DIRECTORY first.")
else:
    print("Enable OPEN_ANNOTATION_WINDOW to mark missing initial TPs.")

## 4. Track every selected image

Enable `RUN_TRACKING` after initial marking. Templates come only from the initial training seeds. Corrections become position anchors; confirming an existing track does not create an appearance template. Previously confirmed points are accepted on reruns only when their coordinates still match. A newly moved proposal needs another review.

In [ ]:
if RUN_TRACKING:
    if not IMAGE_DIRECTORY.is_dir():
        print("Set a valid IMAGE_DIRECTORY first.")
    elif missing_seed_names():
        print("Finish initial TP marking first. Missing:", missing_seed_names())
    else:
        tracking = track_tp(
            config, ANNOTATION_CSV, TRACK_DIRECTORY,
            manual_count=MANUAL_COUNT, patch_size=PATCH_SIZE,
            search_radius=SEARCH_RADIUS, search_roi=TP_SEARCH_ROI,
            min_score=MIN_TRACK_SCORE, min_margin=MIN_PEAK_MARGIN,
            max_templates=MAX_TEMPLATES, overwrite=OVERWRITE_TRACKS,
            review_path=REVIEW_CSV,
        )
        display({k: tracking[k] for k in ("n_frames", "n_uncertain", "tracks_csv")})
else:
    print("Enable RUN_TRACKING after saving all initial annotations.")

## 5. Inspect and review TP positions

The sample below can include all frames or just flagged ones (`FLAGGED_ONLY`). Change `OVERLAY_SEED` for another sample. Accepted correlation matches can still select the wrong feature. Set `REVIEW_ALL_TRACKS=True` to correct those through the same window.

Enable `OPEN_CORRECTION_WINDOW` to review. **A / Accept existing** confirms a correct displayed proposal, including a false alarm. **Click + Enter / Save correction** replaces an incorrect position. **S / Defer** advances without approval; the unresolved point will return next time. Close any time to save progress. Decisions and coordinates are saved in `tp_review.csv` and `tp_manual.csv` respectively.

In [ ]:
if current_tracking_ready():
    rows = read_rows(TRACKS_CSV)
    matching = [r for r in rows if not FLAGGED_ONLY or r["accepted"] != "True"]
    if matching:
        overlays = tp_gallery(config, TRACKS_CSV, examples=9,
                              seed=OVERLAY_SEED, flagged_only=FLAGGED_ONLY)
        display(overlays["figure"])
        plt.close(overlays["figure"])
    else:
        print("No unresolved rows match this gallery selection.")

In [ ]:
if OPEN_CORRECTION_WINDOW:
    if current_tracking_ready():
        launch_tp_annotation(
            config, ANNOTATION_CSV, count=MANUAL_COUNT,
            review_csv=TRACKS_CSV, review_path=REVIEW_CSV,
            review_all=REVIEW_ALL_TRACKS, backend=DESKTOP_BACKEND,
        )
        review_result = apply_tp_review(
            TRACKS_CSV, ANNOTATION_CSV, review_path=REVIEW_CSV,
        )
        display({k: review_result[k] for k in
                 ("n_frames", "n_uncertain", "n_review_decisions_applied")})
else:
    print("Enable OPEN_CORRECTION_WINDOW to inspect and confirm/correct TPs.")

### Apply saved reviews and check readiness

This also recovers saved decisions if you closed the window or interrupted before application. No full tracking rerun is needed. You may optionally rerun tracking to propagate corrected position anchors; inspect any newly changed proposals again. Unresolved points block categories; confirming a false alarm resolves it without changing its coordinates.

In [ ]:
if current_tracking_ready():
    if REVIEW_CSV.is_file():
        applied = apply_tp_review(TRACKS_CSV, ANNOTATION_CSV, review_path=REVIEW_CSV)
        display({k: applied[k] for k in
                 ("n_frames", "n_uncertain", "n_review_decisions_applied")})
    current_tracking_ready(require_resolved=True)

## 6. Save all tracked frames in 16-frame reference sheets

Enable `EXPORT_REFERENCE_SHEETS`. Each PNG contains up to 16 marked frames in a 4×4 grid. The exporter streams images page by page; it does not load the full sequence into memory. At 20,000 frames this produces 1,250 sheets. Thumbnail resizing reduces disk use and visible fine detail; original sensor coordinates remain in the CSV. Sheets can be exported before review to inspect flags; rerun after review to refresh them. If categories already match current tracks, sheets include category labels.

The sheet index saves filenames, marked points, quality/review fields, page, and tile. The sheets are reference visualizations, not quantitative replacements for the TIFFs.

After category export, each frame also shows two horizontal boundaries: **cyan = training mean − SD** (Candidate 3 above), and **yellow = training mean + SD** (Candidate 1 below). Candidate 2 lies between the lines. The fixed thresholds are mapped from original sensor pixels to each thumbnail; TP markers remain visible on top.

In [ ]:
if EXPORT_REFERENCE_SHEETS:
    if current_tracking_ready():
        sheet_source = TRACKS_CSV
        definition_path = CATEGORY_DIRECTORY / "tp_category_definition.json"
        if definition_path.is_file() and (CATEGORY_DIRECTORY / "tp_states.csv").is_file():
            definition = json.loads(definition_path.read_text(encoding="utf-8"))
            if definition.get("tracks_sha256") == hashlib.sha256(TRACKS_CSV.read_bytes()).hexdigest():
                sheet_source = CATEGORY_DIRECTORY / "tp_states.csv"
        sheets = export_tp_montages(
            sheet_source, REFERENCE_DIRECTORY,
            manifest_path=TRACK_DIRECTORY / "tp_tracking_manifest.json",
            tile_width=REFERENCE_TILE_WIDTH,
            overwrite=OVERWRITE_REFERENCE_SHEETS,
        )
        display({k: sheets[k] for k in
                 ("n_frames", "n_pages", "output_directory", "index_csv")})
        page = next(iter(sorted(REFERENCE_DIRECTORY.glob("*.png"))), None)
        if page is not None:
            display(Image(filename=str(page), width=1200))
else:
    print("Enable EXPORT_REFERENCE_SHEETS to save all frames, 16 per sheet.")

## 7. Make the three TP-position categories

Enable `EXPORT_TP_CATEGORIES` once review is complete. Original sensor **y increases downward**:

| Category | Physical meaning | Criterion |
|---|---|---|
| Candidate 1 | Below average band / physically lower | y > training mean + SD |
| Candidate 2 | Average band | mean − SD ≤ y ≤ mean + SD |
| Candidate 3 | Above average band / physically higher | y < training mean − SD |

The band uses population SD (`ddof=0`), calculated only on chronological training frames. Boundaries belong to Candidate 2 and stay fixed for validation/test. These are position bins, not proof of three discrete regimes. No K-means is used.

In [ ]:
if EXPORT_TP_CATEGORIES:
    if current_tracking_ready(require_resolved=True):
        categories = tp_categories(TRACKS_CSV, CATEGORY_DIRECTORY,
                                   overwrite=OVERWRITE_CATEGORIES)
        display(categories)
        display(Image(filename=str(CATEGORY_DIRECTORY / "tp_position_categories.png")))
else:
    print("Enable EXPORT_TP_CATEGORIES once all flagged points are resolved.")

### Inspect original images in each category

After category export, rerun the reference-sheet cell if you want category names on all saved sheets.

In [ ]:
states_path = CATEGORY_DIRECTORY / "tp_states.csv"
if states_path.is_file() and current_tracking_ready(require_resolved=True):
    definition = json.loads((CATEGORY_DIRECTORY / "tp_category_definition.json").read_text())
    if definition["tracks_sha256"] != hashlib.sha256(TRACKS_CSV.read_bytes()).hexdigest():
        print("TP positions changed. Export categories again before inspecting groups.")
    else:
        for label, meaning in [("candidate_1", "Below average band / physically lower"),
                               ("candidate_2", "Average band"),
                               ("candidate_3", "Above average band / physically higher")]:
            print(label, "—", meaning)
            try:
                group = tp_gallery(config, states_path, examples=3,
                                   seed=OVERLAY_SEED, only_category=label)
            except ValueError as exc:
                print(exc)
                continue
            display(group["figure"])
            plt.close(group["figure"])
else:
    print("Export resolved TP categories first.")

## 8. Train POD + RBF-SVM and fidelity-kernel QSVM; evaluate on test

`RUN_POD_AND_CLASSIFIERS=True` is your supplied setting. Processing starts only if reviewed categories exist and match the current TP tracks. Existing nonempty model results are preserved unless `OVERWRITE_POD_RESULTS=True`.

POD is fitted on training images only, then all frames are projected onto that basis. Both classifiers use the same first POD coefficients and TP-derived target labels. Training/validation/test are chronological with gaps. Validation selects model settings; test is evaluated afterward. The default caps are **512 training, 512 validation, 256 test images**, chosen from those blocks, although TP tracking and POD projections cover all selected frames. Increase caps in parameters before tracking for a different experiment; a larger quantum kernel costs more memory/time. QSVM uses a noiseless classical statevector simulation.

In [ ]:
if RUN_POD_AND_CLASSIFIERS:
    category_config = CATEGORY_DIRECTORY / "tp_run_config.json"
    category_definition = CATEGORY_DIRECTORY / "tp_category_definition.json"
    if not category_config.is_file() or not category_definition.is_file():
        print("Export TP categories first; POD/classifiers were not started.")
    elif current_tracking_ready(require_resolved=True):
        definition = json.loads(category_definition.read_text(encoding="utf-8"))
        if definition["tracks_sha256"] != hashlib.sha256(TRACKS_CSV.read_bytes()).hexdigest():
            print("TP positions changed. Export categories again before running models.")
        else:
            tp_run_config = resolve_notebook_config(
                json.loads(category_config.read_text()), PROJECT_DIR,
            )
            # The original spatial-mode figure always uses the parameter value.
            tp_run_config["figures"].update(
                mode_color_vmax=MODE_COLOR_VMAX, mode_color_percentile=None,
            )
            destination = Path(tp_run_config["output_directory"])
            if destination.exists() and any(destination.iterdir()) and not OVERWRITE_POD_RESULTS:
                print("Model results already exist:", destination)
                print("Use the results cell below, or enable OVERWRITE_POD_RESULTS for a new run.")
            else:
                tp_run_manifest = run_from_notebook(
                    tp_run_config, PROJECT_DIR, overwrite=OVERWRITE_POD_RESULTS,
                )
                for source in (category_definition,
                               TRACK_DIRECTORY / "tp_tracking_manifest.json",
                               ANNOTATION_CSV, REVIEW_CSV):
                    if source.is_file():
                        shutil.copyfile(source, destination / source.name)
                display({key: tp_run_manifest.get(key)
                         for key in ("status", "frames", "classification_status")})
else:
    print("Enable RUN_POD_AND_CLASSIFIERS after checking TP categories.")

## 9. See whether each classifier predicts the correct category

The report below exports **only actually evaluated held-out test images**, with TP position, true category, RBF/QSVM prediction, and correctness flags. Blank predictions elsewhere in `state_assignments.csv` are unsampled frames, not classifier failures. Accuracy is agreement with TP-derived labels; its physical meaning depends on measurement accuracy. Do not tune models using these final test outcomes.

In [ ]:
category_config = CATEGORY_DIRECTORY / "tp_run_config.json"
if category_config.is_file() and current_tracking_ready(require_resolved=True):
    result_config = resolve_notebook_config(json.loads(category_config.read_text()), PROJECT_DIR)
    result_directory = Path(result_config["output_directory"])
    if ((result_directory / "classification_metrics.json").is_file()
            and matching_model_results(result_directory, result_config)):
        test_report = export_tp_test_predictions(
            result_directory, CATEGORY_DIRECTORY / "tp_states.csv",
        )
        print(f"Evaluated held-out images: {test_report['n_test']:,}")
        for model, values in test_report["models"].items():
            print(
                f"{model}: accuracy={values['accuracy']:.1%}, "
                f"balanced accuracy={values['balanced_accuracy']:.1%}, "
                f"macro-F1={values['macro_f1']:.3f}"
            )
        print("Per-image predictions:", test_report["predictions_csv"])
        for row in read_rows(test_report["predictions_csv"])[:5]:
            display(row)
        refresh_original_modes(result_directory)
        result_manifest = json.loads((result_directory / "run_manifest.json").read_text())
        display_saved_figures(result_manifest,
                             names=["04_spatial_modes", "09_confusion_matrices"],
                             width=1100)
    else:
        print("No completed classifier metrics yet. Run the preceding stage.")
else:
    print("No TP category experiment is saved yet.")

## Files to keep and checks before publication

- `tp_manual.csv`: manually clicked TP positions in original sensor pixels.
- `tp_review.csv`: explicit human decisions, including accepted false alarms and corrections.
- `tracking/tp_tracks.csv`: complete positions, automatic flags, and review provenance.
- `reference_sheets/`: 16-frame marked PNGs, page/tile index, and export metadata.
- `categories/`: TP states, labels, frozen training thresholds, and plot.
- `categories_pod_qsvm/`: POD outputs, test confusion matrices, numeric metrics, and `tp_test_predictions.csv`.

At 51,032 fps, the first 20 frames span about 0.372 ms. They may not cover the full motion. Inspect accepted as well as flagged matches throughout the record and near extremes. Scores are similarity checks, not calibrated probabilities. Corrections are measurements, not a reason to tune the classifier on test performance. Assess errors near category boundaries and use independent runs where possible. Actual Windows desktop behavior and your experimental tracking accuracy must be checked locally.